# Unified CSBS Scoring Validation Notebook

**Purpose**: Validate both CSBS instruments across two separate REDCap sandboxes:
1. **CSBS Caregiver + EDI-YC**: PID 6205 (NANO - CSBS Caregiver / EDI Scoring Sandbox)
2. **CSBS Baby Siblings (BS)**: PID 6207 (NANO Lab Assessments & Double Data Entry Sandbox)

**Scope**:
- Auto-calculated raw, composite, total, standard score, and percentile score logic matches approved REDCap behavior
- Percentile values are validated with published norms where available and dynamically inferred from populated REDCap records where the source crosswalk is embedded in the study data
- All score fields properly labeled with `AUTO-CALCULATED:` prefix
- Field visibility and highlighting in Online Designer
- Consistency across both instruments (labeling, formula hierarchy, rounding, and norm logic)
- Manual norm fields and any auto-calculated norm companions are scoped to the correct visits

**Tokens Required**:
- `REDCAP_TOKEN_6205`: CSBS Caregiver sandbox (set via environment)
- `REDCAP_TOKEN_6207`: CSBS BS sandbox (defaults to prod token)

**Note**: This is validation-only and does not derive or implement new logic in REDCap itself.

In [1]:
import os
import requests
import math
from decimal import Decimal, ROUND_HALF_UP

API_URL = os.getenv('REDCAP_API_URL', 'https://redcap.research.sc.edu/api/')
TOKEN_6205 = os.getenv('REDCAP_TOKEN_6205', '').strip()  # CSBS Caregiver
TOKEN_6207 = os.getenv('REDCAP_TOKEN_6207', '7B81ED83E91654B3573A2D7FE4B223D2').strip()  # CSBS BS

print('\\n' + '='*70)
print('UNIFIED CSBS VALIDATION - INITIALIZATION')
print('='*70)
print(f'API Endpoint: {API_URL}')
print(f'Token 6205 (Caregiver): {"Set" if TOKEN_6205 else "NOT SET - will skip PID 6205"}')
print(f'Token 6207 (BS): {"Set" if TOKEN_6207 else "NOT SET - cannot proceed"}')

if not TOKEN_6207:
    raise RuntimeError('REDCAP_TOKEN_6207 must be set for CSBS BS validation.')

def post(token, content, **params):
    """Make REDCap API call."""
    data = {'token': token, 'content': content, 'format': 'json', 'returnFormat': 'json'}
    data.update(params)
    r = requests.post(API_URL, data=data, timeout=180)
    r.raise_for_status()
    payload = r.json()
    if isinstance(payload, dict) and payload.get('error'):
        raise RuntimeError(payload['error'])
    return payload

# Verify projects
print('\\n--- Verifying Projects ---')
proj_6207 = post(TOKEN_6207, 'project')
proj_6207 = proj_6207[0] if isinstance(proj_6207, list) and proj_6207 else proj_6207
if str(proj_6207.get('project_id')) != '6207':
    raise RuntimeError('TOKEN_6207 does not map to PID 6207')
print(f'✓ PID 6207: {proj_6207.get("project_title")}')

if TOKEN_6205:
    proj_6205 = post(TOKEN_6205, 'project')
    proj_6205 = proj_6205[0] if isinstance(proj_6205, list) and proj_6205 else proj_6205
    if str(proj_6205.get('project_id')) != '6205':
        raise RuntimeError('TOKEN_6205 does not map to PID 6205')
    print(f'✓ PID 6205: {proj_6205.get("project_title")}')
else:
    print(f'⚠ PID 6205: SKIPPED (token not set)')

\n======================================================================
UNIFIED CSBS VALIDATION - INITIALIZATION
API Endpoint: https://redcap.research.sc.edu/api/
Token 6205 (Caregiver): NOT SET - will skip PID 6205
Token 6207 (BS): Set
\n--- Verifying Projects ---


✓ PID 6207: NANO Lab Assessments & Double Data Entry- Sandbox
⚠ PID 6205: SKIPPED (token not set)


In [2]:
# Utility functions
def num(v):
    """Safe numeric conversion."""
    if v in (None, ''):
        return None
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def half_up(v):
    """Round using ROUND_HALF_UP (REDCap default)."""
    return float(Decimal(str(v)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

def ceil_sum(row, fields, transforms=None):
    """Sum with ceil (CSBS Caregiver domain scoring)."""
    values = [num(row.get(f)) for f in fields]
    if any(v is None for v in values):
        return None
    transforms = transforms or {}
    adjusted = [transforms.get(f, lambda x: x)(v) for f, v in zip(fields, values)]
    return float(math.ceil(sum(adjusted) - 1e-12))

def bs_expected(row):
    """Calculate expected CSBS BS scores."""
    required = [f'csbsbs_scale{i}' for i in range(1, 16)] + [
        'csbsbs_scale16_1', 'csbsbs_scale16_2', 'csbsbs_scale16_3',
        'csbsbs_scale17', 'csbsbs_scale18', 'csbsbs_scale19', 'csbsbs_scale20'
    ]
    if any(num(row.get(k)) is None for k in required):
        return None

    g = lambda name: num(row.get(name))
    emotion = half_up(g('csbsbs_scale1') + g('csbsbs_scale2') + 3 * g('csbsbs_scale3'))
    communication = half_up(g('csbsbs_scale4') / 3 + g('csbsbs_scale5') + g('csbsbs_scale6') + g('csbsbs_scale7'))
    gestures = half_up(2 * g('csbsbs_scale8') + g('csbsbs_scale9'))
    sounds = half_up(g('csbsbs_scale10') + 2 * g('csbsbs_scale11'))
    words = half_up(g('csbsbs_scale12') + g('csbsbs_scale13') / 2 + g('csbsbs_scale14') + g('csbsbs_scale15'))
    understanding = half_up(3 * (g('csbsbs_scale16_1') + g('csbsbs_scale16_2') + g('csbsbs_scale16_3')))
    object_use = half_up(g('csbsbs_scale17') + g('csbsbs_scale18') + g('csbsbs_scale19') + g('csbsbs_scale20'))

    return {
        'csbsbs_emotionraw': emotion,
        'csbsbs_comraw': communication,
        'csbsbs_gesraw': gestures,
        'csbsbs_soundsraw': sounds,
        'csbsbs_wordsraw': words,
        'csbsbs_underraw': understanding,
        'csbsbs_objectraw': object_use,
        'csbsbs_socialcompositecalc': emotion + communication + gestures,
        'csbsbs_speechcompositecalc': sounds + words,
        'csbsbs_symboliccompositecalc': understanding + object_use,
        'csbsbs_totalrawcalc': emotion + communication + gestures + sounds + words + understanding + object_use,
    }

def cg_expected(row):
    """Calculate expected CSBS Caregiver scores."""
    # Field definitions per source notebook
    emotion_fields = [f'csbscg{i}' for i in range(1, 9)]
    communication_fields = [f'csbscg{i}' for i in range(9, 19)]
    gestures_fields = ['csbscg19'] + [f'csbscg20_{i}' for i in range(1, 11)]
    sounds_fields = ['csbscg21', 'csbscg22'] + [f'csbscg23_{i}' for i in range(1, 11)] + ['csbscg24']
    words_fields = ['csbscg25'] + [f'csbscg26_{i}' for i in range(1, 37)] + ['csbscg27', 'csbscg28']
    understanding_fields = ['csbscg29', 'csbscg30', 'csbscg31'] + [f'csbscg32_{i}' for i in range(1, 37)]
    object_fields = ['csbscg33', 'csbscg34'] + [f'csbscg35_{i}' for i in range(1, 11)] + ['csbscg36'] + [f'csbscg37_{i}' for i in range(1, 9)] + ['csbscg38', 'csbscg39'] + [f'csbscg40_{i}' for i in range(1, 11)] + [f'csbscg41_{i}' for i in range(1, 7)]
    
    # Half-weighted fields in various domains
    half_fields = {f: (lambda x: x / 2) for f in words_fields + understanding_fields + object_fields 
                   if f.split('_')[0][5:] in ('26', '32', '35', '37', '40', '41')}
    
    # Special transform for csbscg5: (2 - value)
    emotion_transforms = {'csbscg5': lambda x: 2 - x}
    
    emotion = ceil_sum(row, emotion_fields, emotion_transforms)
    communication = ceil_sum(row, communication_fields)
    gestures = ceil_sum(row, gestures_fields)
    sounds = ceil_sum(row, sounds_fields)
    words = ceil_sum(row, words_fields, half_fields)
    understanding = ceil_sum(row, understanding_fields, half_fields)
    object_use = ceil_sum(row, object_fields, half_fields)
    
    if any(v is None for v in [emotion, communication, gestures, sounds, words, understanding, object_use]):
        return None
    
    social = emotion + communication + gestures
    speech = sounds + words
    symbolic = understanding + object_use
    
    return {
        'csbs_emotionandeyegaze': emotion,
        'csbs_communication': communication,
        'csbs_gestures': gestures,
        'csbs_sounds': sounds,
        'csbs_words': words,
        'csbs_understanding': understanding,
        'csbs_objectuse': object_use,
        'csbs_socialcomposite': social,
        'csbs_speechcomposite': speech,
        'csbs_symboliccomposite': symbolic,
        'cbscg_totalscore': social + speech + symbolic,
    }

print('Utility functions and scoring logic loaded.')

Utility functions and scoring logic loaded.


In [3]:
import re
from collections import Counter, defaultdict

CG_SCALE_PERCENTILE = {
    17: 99, 16: 98, 15: 95, 14: 91, 13: 84, 12: 75, 11: 63,
    10: 50, 9: 37, 8: 25, 7: 16, 6: 9, 5: 5, 4: 2, 3: 1,
}
CG_TOTAL_PERCENTILE = {
    135: 99, 134: 99, 133: 99, 132: 98, 131: 98, 130: 98,
    129: 97, 128: 97, 127: 96, 126: 96, 125: 95, 124: 95,
    123: 94, 122: 93, 121: 92, 120: 91, 119: 90, 118: 89,
    117: 87, 116: 86, 115: 84, 114: 83, 113: 81, 112: 79,
    111: 77, 110: 75, 109: 73, 108: 70, 107: 68, 106: 66,
    105: 63, 104: 61, 103: 58, 102: 55, 101: 53, 100: 50,
    99: 47, 98: 45, 97: 42, 96: 40, 95: 37, 94: 35, 93: 32,
    92: 30, 91: 27, 90: 25, 89: 23, 88: 21, 87: 19, 86: 18,
    85: 16, 84: 14, 83: 13, 82: 12, 81: 10, 80: 9, 79: 8,
    78: 7, 77: 6, 76: 6, 75: 5, 74: 4, 73: 4, 72: 3, 71: 3,
    70: 2, 69: 2, 68: 2, 67: 1, 66: 1, 65: 1,
}

CG_NORMS_23_24 = {
    'csbs_emotionandeyegaze': [(16,16,17),(15,15,14),(14,14,11),(13,13,9),(12,12,7),(11,11,5),(10,10,4),(0,9,3)],
    'csbs_communication': [(20,20,17),(19,19,12),(18,18,10),(16,17,9),(15,15,8),(12,14,6),(10,11,5),(9,9,4),(0,8,3)],
    'csbs_gestures': [(12,12,17),(11,11,9),(10,10,6),(9,9,5),(7,8,4),(0,6,3)],
    'csbs_sounds': [(16,16,17),(15,15,11),(14,14,10),(12,13,9),(11,11,8),(9,10,7),(7,8,6),(6,6,5),(0,5,3)],
    'csbs_words': [(24,24,17),(23,23,12),(20,22,11),(17,19,10),(13,16,9),(10,12,8),(8,9,7),(4,7,6),(3,3,5),(2,2,4),(0,1,3)],
    'csbs_understanding': [(24,24,17),(23,23,10),(22,22,9),(20,21,8),(15,19,7),(12,14,6),(7,11,5),(6,6,4),(0,5,3)],
    'csbs_objectuse': [(27,27,17),(25,26,12),(23,24,11),(22,22,10),(20,21,9),(18,19,8),(16,17,7),(14,15,6),(13,13,5),(10,12,4),(0,9,3)],
    'csbs_socialcomposite': [(48,48,17),(47,47,15),(46,46,14),(45,45,12),(43,44,11),(42,42,10),(40,41,9),(39,39,8),(38,38,7),(35,37,6),(34,34,5),(27,33,4),(0,26,3)],
    'csbs_speechcomposite': [(40,40,17),(39,39,13),(37,38,12),(33,36,11),(30,32,10),(25,29,9),(21,24,8),(16,20,7),(12,15,6),(10,11,5),(9,9,4),(0,8,3)],
    'csbs_symboliccomposite': [(51,51,17),(50,50,13),(49,49,12),(46,48,11),(44,45,10),(41,43,9),(36,40,8),(34,35,7),(27,33,6),(24,26,5),(17,23,4),(0,16,3)],
    'cbscg_totalscore': [(139,139,135),(138,138,131),(136,137,125),(135,135,124),(134,134,118),(133,133,115),(132,132,114),(130,131,112),(129,129,111),(128,128,110),(127,127,108),(126,126,106),(125,125,105),(124,124,104),(122,123,103),(121,121,102),(120,120,101),(119,119,100),(118,118,99),(117,117,98),(116,116,97),(115,115,96),(114,114,95),(112,113,94),(110,111,93),(109,109,92),(106,108,91),(104,105,90),(102,103,89),(100,101,88),(99,99,87),(98,98,86),(96,97,85),(94,95,84),(93,93,83),(88,92,82),(87,87,81),(84,86,80),(82,83,79),(80,81,78),(79,79,77),(77,78,76),(75,76,75),(74,74,74),(73,73,73),(72,72,72),(71,71,71),(61,70,70),(59,60,69),(56,58,68),(53,55,67),(50,52,66),(0,49,65)],
}

CG_RAW_MAX = {
    'csbs_emotionandeyegaze': 16, 'csbs_communication': 20, 'csbs_gestures': 12,
    'csbs_sounds': 16, 'csbs_words': 24, 'csbs_understanding': 24,
    'csbs_objectuse': 27, 'csbs_socialcomposite': 48, 'csbs_speechcomposite': 40,
    'csbs_symboliccomposite': 51, 'cbscg_totalscore': 139,
}

CG_NORM_FIELDS = {
    'csbs_emotionandeyegaze': ('Emotion & eye gaze', 'csbs_emotionss', 'csbs_emotionandeyegazeper'),
    'csbs_communication': ('Communication', 'csbs_communicationss', 'csbs_communicationper'),
    'csbs_gestures': ('Gestures', 'csbs_gesturesss', 'csbs_gesturesper'),
    'csbs_sounds': ('Sounds', 'csbs_soundsss', 'csbs_soundsper'),
    'csbs_words': ('Words', 'csbs_wordsss', 'csbs_wordsper'),
    'csbs_understanding': ('Understanding', 'csbs_understandingss', 'csbs_understandingper'),
    'csbs_objectuse': ('Object use', 'csbs_objectusess', 'csbs_objectuseper'),
    'csbs_socialcomposite': ('Social composite', 'csbs_socialcompositess', 'csbs_socialcompositeper'),
    'csbs_speechcomposite': ('Speech composite', 'csbs_speechcompositess', 'csbs_speechcompositeper'),
    'csbs_symboliccomposite': ('Symbolic composite', 'csbs_symboliccompositess', 'csbs_symboliccompositeper'),
    'cbscg_totalscore': ('Total', 'csbs_totalss', 'csbs_totalper'),
}

BS_NORM_GROUPS = {
    'csbsbs_emotionraw': {
        'label': 'Emotion and Eye Gaze',
        'targets': {
            'standard': ['csbsbs_emotionstandard', 'csbsbs_emotionstandardpre'],
            'percentile': ['csbsbs_emotionpercentile', 'csbsbs_emotionpercentilepre'],
        },
    },
    'csbsbs_comraw': {
        'label': 'Communication',
        'targets': {
            'standard': ['csbsbs_comstandard', 'csbsbs_comstandardpre'],
            'percentile': ['csbsbs_compercentile', 'csbsbs_compercentilepre'],
        },
    },
    'csbsbs_gesraw': {
        'label': 'Gestures',
        'targets': {
            'standard': ['csbsbs_gesturesstandard', 'csbsbs_gesturesstandardpre'],
            'percentile': ['csbsbs_gesturespercentile', 'csbsbs_gesturespercentilepre'],
        },
    },
    'csbsbs_soundsraw': {
        'label': 'Sounds',
        'targets': {
            'standard': ['csbsbs_soundsstandard', 'csbsbs_soundsstandardpre'],
            'percentile': ['csbsbs_soundspercentile', 'csbsbs_soundspercentilepre'],
        },
    },
    'csbsbs_wordsraw': {
        'label': 'Words',
        'targets': {
            'standard': ['csbsbs_wordsstandard', 'csbsbs_wordsstandardpre'],
            'percentile': ['csbsbs_wordspercentile', 'csbsbs_wordspercentilepre'],
        },
    },
    'csbsbs_underraw': {
        'label': 'Understanding',
        'targets': {
            'standard': ['csbsbs_understandingstandard', 'csbsbs_understandingstandardpre'],
            'percentile': ['csbsbs_understandingpercentile', 'csbsbs_understandingpercentilepre'],
        },
    },
    'csbsbs_objectraw': {
        'label': 'Object Use',
        'targets': {
            'standard': ['csbsbs_objectstandard', 'csbsbs_objectstandardpre'],
            'percentile': ['csbsbs_objectpercentile', 'csbsbs_objectpercentilepre'],
        },
    },
    'csbsbs_socialcompositecalc': {
        'label': 'Social',
        'targets': {
            'standard': ['csbsbs_socialstandard', 'csbsbs_socialstandardpre'],
            'percentile': ['csbsbs_socialpercentile', 'csbsbs_socialpercentilepre'],
        },
    },
    'csbsbs_speechcompositecalc': {
        'label': 'Speech',
        'targets': {
            'standard': ['csbsbs_speechstandard', 'csbsbs_speechstandardpre'],
            'percentile': ['csbsbs_speechpercentile', 'csbsbs_speechpercentilepre'],
        },
    },
    'csbsbs_symboliccompositecalc': {
        'label': 'Symbolic',
        'targets': {
            'standard': ['csbsbs_symbolicstandard', 'csbsbs_symbolicstandardpre'],
            'percentile': ['csbsbs_symbolicpercentile', 'csbsbs_symbolicpercentilepre'],
        },
    },
    'csbsbs_totalrawcalc': {
        'label': 'Total',
        'targets': {
            'standard': ['csbsbs_total', 'csbsbs_totalstandardpre'],
            'percentile': ['csbsbs_totalpercent', 'csbsbs_totalpercentilepre'],
        },
    },
}


def validate_cg_norm_domains():
    for field, maximum in CG_RAW_MAX.items():
        covered = [raw for low, high, _ in CG_NORMS_23_24[field] for raw in range(low, high + 1)]
        if sorted(covered) != list(range(maximum + 1)) or len(covered) != len(set(covered)):
            raise ValueError(f'Incomplete or overlapping 23-24 month norm table: {field}')


validate_cg_norm_domains()


def build_numeric_lookup(records, raw_getter, target_field):
    counts = defaultdict(Counter)
    samples = 0
    for row in records:
        raw_value = raw_getter(row)
        target_value = num(row.get(target_field))
        if raw_value is None or target_value is None:
            continue
        counts[int(round(raw_value))][int(round(target_value))] += 1
        samples += 1
    lookup = {raw: counter.most_common(1)[0][0] for raw, counter in counts.items() if counter}
    conflicts = {raw: dict(counter) for raw, counter in counts.items() if len(counter) > 1}
    return lookup, conflicts, samples


def render_redcap_if_formula(raw_field, mapping, default=''):
    expr = '""' if default == '' else str(default)
    for raw_value, target_value in sorted(mapping.items(), reverse=True):
        expr = f'if([{raw_field}]={raw_value},{target_value},{expr})'
    return expr


def cg_norm(raw_field, raw_value):
    if raw_value is None:
        return None, None
    raw_value = int(round(raw_value))
    for low, high, standard in CG_NORMS_23_24[raw_field]:
        if low <= raw_value <= high:
            percentile = CG_TOTAL_PERCENTILE[standard] if raw_field == 'cbscg_totalscore' else CG_SCALE_PERCENTILE[standard]
            return standard, percentile
    return None, None


## CSBS BS (PID 6207) Validation

In [4]:
print('\n' + '='*70)
print('CSBS BS (PID 6207) - VALIDATION')
print('='*70)

metadata = post(TOKEN_6207, 'metadata')
record_id_field = metadata[0]['field_name']
score_fields = [
    'csbsbs_emotionraw','csbsbs_comraw','csbsbs_gesraw','csbsbs_soundsraw','csbsbs_wordsraw',
    'csbsbs_underraw','csbsbs_objectraw','csbsbs_socialcompositecalc','csbsbs_speechcompositecalc',
    'csbsbs_symboliccompositecalc','csbsbs_totalrawcalc'
]
input_fields = [f'csbsbs_scale{i}' for i in range(1,16)] + [
    'csbsbs_scale16_1','csbsbs_scale16_2','csbsbs_scale16_3','csbsbs_scale17','csbsbs_scale18','csbsbs_scale19','csbsbs_scale20'
]
records = post(TOKEN_6207, 'record', type='flat', rawOrLabel='raw', rawOrLabelHeaders='raw')

comparisons = 0
mismatches = []
for row in records:
    exp = bs_expected(row)
    if exp is None:
        continue
    rid = row.get(record_id_field, '')
    ev = row.get('redcap_event_name', '')
    for f, expected in exp.items():
        actual = num(row.get(f))
        if actual is None:
            mismatches.append({'record': rid, 'event': ev, 'field': f, 'actual': 'MISSING', 'expected': expected})
            continue
        comparisons += 1
        if abs(actual - expected) > 1e-9:
            mismatches.append({'record': rid, 'event': ev, 'field': f, 'actual': actual, 'expected': expected})

print(f'\nRecords analyzed: {len(records)}')
print(f'Score comparisons: {comparisons}')
print(f'Mismatches found: {len(mismatches)}')

if mismatches:
    print('\n⚠ MISMATCHES DETAIL (first 10):')
    for m in mismatches[:10]:
        print(f"  {m['record']}/{m['event']}/{m['field']}: {m['actual']} vs {m['expected']}")
    bs_validation_status = 'REVIEW REQUIRED'
else:
    print('\n✓ PASS: All CSBS BS scores match approved logic')
    bs_validation_status = 'PASS'



CSBS BS (PID 6207) - VALIDATION



Records analyzed: 1767
Score comparisons: 5533
Mismatches found: 0

✓ PASS: All CSBS BS scores match approved logic


In [5]:
print('\n' + '='*70)
print('CSBS BS STANDARD / PERCENTILE AUDIT')
print('='*70)

bs_expected_rows_by_event = defaultdict(list)
for row in records:
    expected = bs_expected(row)
    if expected is not None:
        bs_expected_rows_by_event[row.get('redcap_event_name', '')].append((row, expected))


def build_lookup_from_expected(event_rows, raw_field, target_field):
    counts = defaultdict(Counter)
    samples = 0
    for row, expected in event_rows:
        raw_value = expected.get(raw_field)
        target_value = num(row.get(target_field))
        if raw_value is None or target_value is None:
            continue
        counts[int(round(raw_value))][int(round(target_value))] += 1
        samples += 1
    lookup = {raw: counter.most_common(1)[0][0] for raw, counter in counts.items() if counter}
    conflicts = {raw: dict(counter) for raw, counter in counts.items() if len(counter) > 1}
    return lookup, conflicts, samples


bs_norm_issues = []
bs_norm_conflicts = []
bs_norm_lookup_maps = {}

for event_name, event_rows in bs_expected_rows_by_event.items():
    for raw_field, spec in BS_NORM_GROUPS.items():
        for kind, target_fields in spec['targets'].items():
            for target_field in target_fields:
                lookup, conflicts, samples = build_lookup_from_expected(event_rows, raw_field, target_field)
                if lookup:
                    bs_norm_lookup_maps[(event_name, target_field)] = lookup
                if conflicts:
                    bs_norm_conflicts.append((event_name, raw_field, target_field, conflicts))
                if not samples:
                    continue

                for row, expected in event_rows:
                    raw_value = expected.get(raw_field)
                    actual_value = row.get(target_field)
                    if raw_value is None or actual_value in (None, ''):
                        continue
                    expected_value = lookup.get(int(round(raw_value)))
                    actual_numeric = num(actual_value)
                    if expected_value is None:
                        bs_norm_issues.append((event_name, row.get(record_id_field, ''), raw_field, target_field, actual_value, 'NO LOOKUP'))
                        continue
                    if actual_numeric is None or abs(actual_numeric - expected_value) > 1e-9:
                        bs_norm_issues.append((event_name, row.get(record_id_field, ''), raw_field, target_field, actual_value, expected_value))

print(f'\nEvents analyzed for norms: {len(bs_expected_rows_by_event)}')
print(f'Norm mismatches: {len(bs_norm_issues)}')
print(f'Conflicting event-specific raw->target mappings: {len(bs_norm_conflicts)}')

if bs_norm_conflicts:
    print('\n⚠ Conflicts found (first 5):')
    for event_name, raw_field, target_field, conflicts in bs_norm_conflicts[:5]:
        print(f'  {event_name} :: {raw_field} -> {target_field}: {conflicts}')

if bs_norm_issues:
    print('\n⚠ Norm mismatches (first 10):')
    for item in bs_norm_issues[:10]:
        print(f'  {item[0]} / {item[1]} / {item[2]} -> {item[3]}: {item[4]} vs {item[5]}')
    bs_percentile_status = 'REVIEW REQUIRED'
else:
    print('\n✓ PASS: All BS standard and percentile fields match event-specific REDCap logic')
    bs_percentile_status = 'PASS'

print('\n--- Proposed auto-calculated percentile fields ---')
for event_name, event_rows in bs_expected_rows_by_event.items():
    example_row = event_rows[0][0]
    print(f'  Event: {event_name}')
    for raw_field, spec in BS_NORM_GROUPS.items():
        percentile_field = spec['targets']['percentile'][0]
        lookup = bs_norm_lookup_maps.get((event_name, percentile_field))
        if not lookup:
            continue
        formula = render_redcap_if_formula(raw_field, lookup)
        print(f'    {percentile_field}_auto <= {raw_field}: {formula[:120]}{"..." if len(formula) > 120 else ""}')



CSBS BS STANDARD / PERCENTILE AUDIT

Events analyzed for norms: 2
Norm mismatches: 564
Conflicting event-specific raw->target mappings: 41

⚠ Conflicts found (first 5):
  9_months_arm_1 :: csbsbs_emotionraw -> csbsbs_emotionstandard: {14: {10: 3, 14: 2}, 12: {9: 3, 12: 1}, 8: {7: 2, 8: 1}}
  9_months_arm_1 :: csbsbs_comraw -> csbsbs_comstandard: {12: {7: 3, 12: 1}, 3: {3: 2, 4: 2}}
  9_months_arm_1 :: csbsbs_gesraw -> csbsbs_gesturesstandard: {0: {3: 6, 0: 1}, 2: {2: 2, 4: 1}}
  9_months_arm_1 :: csbsbs_soundsraw -> csbsbs_soundsstandard: {0: {7: 7, 0: 4}, 3: {7: 4, 3: 3}}
  9_months_arm_1 :: csbsbs_wordsraw -> csbsbs_wordsstandard: {0: {10: 11, 0: 9}}

⚠ Norm mismatches (first 10):
  9_months_arm_1 / 5098--2 / csbsbs_emotionraw -> csbsbs_emotionstandard: 14 vs 10
  9_months_arm_1 / 5100 / csbsbs_emotionraw -> csbsbs_emotionstandard: 14 vs 10
  9_months_arm_1 / 5104 / csbsbs_emotionraw -> csbsbs_emotionstandard: 12 vs 9
  9_months_arm_1 / 5111--2 / csbsbs_emotionraw -> csbsbs_emotions

## CSBS Caregiver (PID 6205) Validation

In [6]:
if TOKEN_6205:
    print('\n' + '='*70)
    print('CSBS Caregiver (PID 6205) - VALIDATION')
    print('='*70)
    
    metadata_6205 = post(TOKEN_6205, 'metadata')
    record_id_field_6205 = metadata_6205[0]['field_name']
    cg_score_fields = [
        'csbs_emotionandeyegaze','csbs_communication','csbs_gestures','csbs_sounds',
        'csbs_words','csbs_understanding','csbs_objectuse','csbs_socialcomposite',
        'csbs_speechcomposite','csbs_symboliccomposite','cbscg_totalscore'
    ]
    
    records_6205 = post(TOKEN_6205, 'record', type='flat', rawOrLabel='raw', rawOrLabelHeaders='raw')
    
    comparisons_6205 = 0
    mismatches_6205 = []
    for row in records_6205:
        exp = cg_expected(row)
        if exp is None:
            continue
        rid = row.get(record_id_field_6205, '')
        ev = row.get('redcap_event_name', '')
        for f, expected in exp.items():
            actual = num(row.get(f))
            if actual is None:
                mismatches_6205.append({'record': rid, 'event': ev, 'field': f, 'actual': 'MISSING', 'expected': expected})
                continue
            comparisons_6205 += 1
            if abs(actual - expected) > 1e-9:
                mismatches_6205.append({'record': rid, 'event': ev, 'field': f, 'actual': actual, 'expected': expected})
    
    print(f'\nRecords analyzed: {len(records_6205)}')
    print(f'Score comparisons: {comparisons_6205}')
    print(f'Mismatches found: {len(mismatches_6205)}')
    
    if mismatches_6205:
        print('\n⚠ MISMATCHES DETAIL (first 10):')
        for m in mismatches_6205[:10]:
            print(f"  {m['record']}/{m['event']}/{m['field']}: {m['actual']} vs {m['expected']}")
        cg_validation_status = 'REVIEW REQUIRED'
    else:
        print('\n✓ PASS: All CSBS Caregiver scores match approved logic')
        cg_validation_status = 'PASS'
else:
    print('\n⚠ CSBS Caregiver (PID 6205): SKIPPED (token not set)')
    cg_validation_status = 'SKIPPED'



⚠ CSBS Caregiver (PID 6205): SKIPPED (token not set)


In [7]:
if TOKEN_6205:
    print('\n' + '='*70)
    print('CSBS CAREGIVER STANDARD / PERCENTILE AUDIT')
    print('='*70)

    cg_expected_rows = []
    for row in records_6205:
        expected = cg_expected(row)
        if expected is not None:
            cg_expected_rows.append((row, expected))

    cg_norm_issues = []
    cg_norm_conflicts = []

    for raw_field, (_, ss_field, percentile_field) in CG_NORM_FIELDS.items():
        standard_lookup = {raw: cg_norm(raw_field, raw)[0] for raw in range(CG_RAW_MAX[raw_field] + 1) if cg_norm(raw_field, raw)[0] is not None}
        percentile_lookup = {raw: cg_norm(raw_field, raw)[1] for raw in range(CG_RAW_MAX[raw_field] + 1) if cg_norm(raw_field, raw)[1] is not None}

        for target_field, lookup in ((ss_field, standard_lookup), (percentile_field, percentile_lookup)):
            for row, expected in cg_expected_rows:
                raw_value = expected.get(raw_field)
                actual_value = row.get(target_field)
                if raw_value is None or actual_value in (None, ''):
                    continue
                expected_value = lookup.get(int(round(raw_value)))
                actual_numeric = num(actual_value)
                if expected_value is None:
                    cg_norm_issues.append((row.get(record_id_field_6205, ''), row.get('redcap_event_name', ''), raw_field, target_field, actual_value, 'NO LOOKUP'))
                    continue
                if actual_numeric is None or abs(actual_numeric - expected_value) > 1e-9:
                    cg_norm_issues.append((row.get(record_id_field_6205, ''), row.get('redcap_event_name', ''), raw_field, target_field, actual_value, expected_value))

    total_standard_lookup = {raw: cg_norm('cbscg_totalscore', raw)[0] for raw in range(CG_RAW_MAX['cbscg_totalscore'] + 1) if cg_norm('cbscg_totalscore', raw)[0] is not None}
    total_percentile_lookup = {raw: cg_norm('cbscg_totalscore', raw)[1] for raw in range(CG_RAW_MAX['cbscg_totalscore'] + 1) if cg_norm('cbscg_totalscore', raw)[1] is not None}

    print(f'\nRecords analyzed for norms: {len(cg_expected_rows)}')
    print(f'Norm mismatches: {len(cg_norm_issues)}')

    if cg_norm_issues:
        print('\n⚠ Norm mismatches (first 10):')
        for item in cg_norm_issues[:10]:
            print(f'  {item[0]}/{item[1]}/{item[2]} -> {item[3]}: {item[4]} vs {item[5]}')
        cg_percentile_status = 'REVIEW REQUIRED'
    else:
        print('\n✓ PASS: All CSBS Caregiver standard and percentile fields match published norm tables')
        cg_percentile_status = 'PASS'

    print('\n--- Proposed auto-calculated total norm fields ---')
    print(f"  csbs_totalss_auto <= cbscg_totalscore: {render_redcap_if_formula('cbscg_totalscore', total_standard_lookup)[:160]}{'...' if len(render_redcap_if_formula('cbscg_totalscore', total_standard_lookup)) > 160 else ''}")
    print(f"  csbs_totalper_auto <= cbscg_totalscore: {render_redcap_if_formula('cbscg_totalscore', total_percentile_lookup)[:160]}{'...' if len(render_redcap_if_formula('cbscg_totalscore', total_percentile_lookup)) > 160 else ''}")
else:
    print('\n⚠ CSBS Caregiver (PID 6205): SKIPPED (token not set)')
    cg_percentile_status = 'SKIPPED'



⚠ CSBS Caregiver (PID 6205): SKIPPED (token not set)


## Field Labels & Visibility Audit

In [8]:
print('\\n' + '='*70)
print('FIELD LABELS AUDIT - AUTO-CALCULATED Prefix')
print('='*70)

print('\\n--- CSBS BS (PID 6207) ---')
metadata_6207 = post(TOKEN_6207, 'metadata')
bs_by_field = {x['field_name']: x for x in metadata_6207}
bs_score_fields = [
    'csbsbs_emotionraw','csbsbs_comraw','csbsbs_gesraw','csbsbs_soundsraw','csbsbs_wordsraw',
    'csbsbs_underraw','csbsbs_objectraw','csbsbs_socialcompositecalc','csbsbs_speechcompositecalc',
    'csbsbs_symboliccompositecalc','csbsbs_totalrawcalc'
]

bs_label_issues = []
for f in bs_score_fields:
    meta = bs_by_field.get(f)
    if not meta:
        bs_label_issues.append((f, 'FIELD NOT FOUND'))
    else:
        label = meta.get('field_label', '')
        ftype = meta.get('field_type', '')
        has_auto = label.startswith('AUTO-CALCULATED:')
        if has_auto and ftype == 'calc':
            print(f'  ✓ {f:<30} : {label[:60]}')
        else:
            issue = f'Missing AUTO-CALCULATED prefix' if not has_auto else f'Type={ftype} (expected calc)'
            bs_label_issues.append((f, issue))

if bs_label_issues:
    print('\\n  ⚠ Issues found:')
    for f, issue in bs_label_issues:
        print(f'    {f}: {issue}')
else:
    print('\\n  ✓ All CSBS BS score fields properly labeled with AUTO-CALCULATED prefix')

if TOKEN_6205:
    print('\\n--- CSBS Caregiver (PID 6205) ---')
    cg_by_field = {x['field_name']: x for x in metadata_6205}
    cg_score_fields = [
        'csbs_emotionandeyegaze','csbs_communication','csbs_gestures','csbs_sounds',
        'csbs_words','csbs_understanding','csbs_objectuse','csbs_socialcomposite',
        'csbs_speechcomposite','csbs_symboliccomposite','cbscg_totalscore'
    ]
    
    cg_label_issues = []
    for f in cg_score_fields:
        meta = cg_by_field.get(f)
        if not meta:
            cg_label_issues.append((f, 'FIELD NOT FOUND'))
        else:
            label = meta.get('field_label', '')
            ftype = meta.get('field_type', '')
            has_auto = label.startswith('AUTO-CALCULATED:')
            if has_auto and ftype == 'calc':
                print(f'  ✓ {f:<30} : {label[:60]}')
            else:
                issue = f'Missing AUTO-CALCULATED prefix' if not has_auto else f'Type={ftype} (expected calc)'
                cg_label_issues.append((f, issue))
    
    if cg_label_issues:
        print('\\n  ⚠ Issues found:')
        for f, issue in cg_label_issues:
            print(f'    {f}: {issue}')
    else:
        print('\\n  ✓ All CSBS Caregiver score fields properly labeled with AUTO-CALCULATED prefix')

\n======================================================================
FIELD LABELS AUDIT - AUTO-CALCULATED Prefix
\n--- CSBS BS (PID 6207) ---


  ✓ csbsbs_emotionraw              : AUTO-CALCULATED: Emotion and Eye Gaze Weighted Raw Score
  ✓ csbsbs_comraw                  : AUTO-CALCULATED: Communication Weighted Raw Score
  ✓ csbsbs_gesraw                  : AUTO-CALCULATED: Gestures Weighted Raw Score
  ✓ csbsbs_soundsraw               : AUTO-CALCULATED: Sounds Weighted Raw Score
  ✓ csbsbs_wordsraw                : AUTO-CALCULATED: Words Weighted Raw Score
  ✓ csbsbs_underraw                : AUTO-CALCULATED: Understanding Weighted Raw Score
  ✓ csbsbs_objectraw               : AUTO-CALCULATED: Object Use Weighted Raw Score
  ✓ csbsbs_socialcompositecalc     : AUTO-CALCULATED: Social Composite Score
  ✓ csbsbs_speechcompositecalc     : AUTO-CALCULATED: Speech Composite Score
  ✓ csbsbs_symboliccompositecalc   : AUTO-CALCULATED: Symbolic Composite Score
  ✓ csbsbs_totalrawcalc            : AUTO-CALCULATED: CSBS BS Total Raw Score
\n  ✓ All CSBS BS score fields properly labeled with AUTO-CALCULATED prefix


## Consistency Check & Summary

In [9]:
print('\n' + '='*70)
print('COMPREHENSIVE VALIDATION SUMMARY')
print('='*70)

print('\n1. SCORING LOGIC VALIDATION:')
print(f'   CSBS BS (PID 6207):           {bs_validation_status}')
print(f'   CSBS BS Percentiles:           {bs_percentile_status}')
if TOKEN_6205:
    print(f'   CSBS Caregiver (PID 6205):    {cg_validation_status}')
    print(f'   CSBS Caregiver Percentiles:    {cg_percentile_status}')
else:
    print(f'   CSBS Caregiver (PID 6205):    ⚠ {cg_validation_status}')
    print(f'   CSBS Caregiver Percentiles:    ⚠ {cg_percentile_status}')

print('\n2. FIELD LABELING:')
if not bs_label_issues:
    print(f'   CSBS BS (PID 6207):           ✓ All fields properly labeled')
else:
    print(f'   CSBS BS (PID 6207):           ⚠ {len(bs_label_issues)} issues')

if TOKEN_6205:
    if not cg_label_issues:
        print(f'   CSBS Caregiver (PID 6205):    ✓ All fields properly labeled')
    else:
        print(f'   CSBS Caregiver (PID 6205):    ⚠ {len(cg_label_issues)} issues')

print('\n3. RECORD COVERAGE:')
print(f'   CSBS BS: {len(records)} records analyzed')
if TOKEN_6205:
    print(f'   CSBS Caregiver: {len(records_6205)} records analyzed')

print('\n4. AUTO-FIELD COVERAGE:')
print('   BS percentile companions: proposed from dynamic REDCap inference')
print('   Caregiver total companions: proposed from published 23-24 month norm tables')

print('\n' + '='*70)
if TOKEN_6205:
    overall = 'PASS' if (bs_validation_status == 'PASS' and bs_percentile_status == 'PASS' and cg_validation_status == 'PASS' and cg_percentile_status == 'PASS' and not bs_label_issues and not cg_label_issues) else 'REVIEW REQUIRED'
else:
    overall = 'PARTIAL' if bs_validation_status == 'PASS' and bs_percentile_status == 'PASS' and not bs_label_issues else 'REVIEW REQUIRED'

print(f'OVERALL VALIDATION STATUS: {overall}')
if not TOKEN_6205:
    print('\nTo complete full dual-project validation, set:')
    print('  export REDCAP_TOKEN_6205=<your_token>')
print('='*70)



COMPREHENSIVE VALIDATION SUMMARY

1. SCORING LOGIC VALIDATION:
   CSBS BS (PID 6207):           PASS
   CSBS BS Percentiles:           REVIEW REQUIRED
   CSBS Caregiver (PID 6205):    ⚠ SKIPPED
   CSBS Caregiver Percentiles:    ⚠ SKIPPED

2. FIELD LABELING:
   CSBS BS (PID 6207):           ✓ All fields properly labeled

3. RECORD COVERAGE:
   CSBS BS: 1767 records analyzed

4. AUTO-FIELD COVERAGE:
   BS percentile companions: proposed from dynamic REDCap inference
   Caregiver total companions: proposed from published 23-24 month norm tables

OVERALL VALIDATION STATUS: REVIEW REQUIRED

To complete full dual-project validation, set:
  export REDCAP_TOKEN_6205=<your_token>
